# CPU域孪生调试

在开发Ascend C核函数的过程中，开发者可以首先在非昇腾设备上利用CPU仿真环境进行算子的开发和测试，待功能验证通过后，再无缝迁移至昇腾设备进行加速计算。本章节将重点介绍完成Ascend C核函数编写后，如何编写CPU侧的调用程序，并通过printf打印的方式进行CPU域调试。这有助于在没有昇腾硬件的情况下快速验证算子的正确性，为后续在真实设备上运行打下坚实基础。

本节学习大纲如下：

- CPU侧验证程序的开发与调用方法
- 基于printf的CPU域调试实践


---
## 1. 环境准备

正式开始学习之前，先要对jupyter环境进行初始化。以下代码完成了初始化并将环境中的变量导入jupyter环境，同时完成了代码目录的创建。保证能正常导入代码以及使用Bisheng编译器，完成算子的开发及编译。

In [ ]:
!mkdir -p Sources/07.02

import os, subprocess
env = subprocess.check_output("bash -l -c 'source $ASCEND_TOOLKIT_HOME/set_env.sh && env'", shell=True, text=True)
for line in env.splitlines():
    if "=" in line: os.environ.__setitem__(*line.split("=", 1))
print("\n🎉 Environment initialization process completed successfully!")

---

## 2. CPU侧验证程序的开发与调用方法

在非昇腾设备上，开发者可以利用CPU仿真环境先行进行算子开发和测试，并在准备就绪后，利用昇腾设备进行加速计算。在[基于Add算子的核函数介绍](../02_AscendC_basic/02.04_introduction_to_kernel_functions_based_on_add_operator.ipynb)章节，我们已经介绍了算子Kernel程序NPU域的编译运行。相比于NPU域的算子运行逻辑，CPU域调试，实际上是通过标准的GCC编译器编译算子Kernel程序。此时算子Kernel程序链接CPU调测库，执行编译生成的可执行文件，可以完成算子CPU域的运行验证。CPU域和NPU域的核函数运行逻辑对比如下图所示：

<img src="./images/CPU_NPU_Kernel_Function_Execution_Logic_Comparison.png" alt="CPU_NPU_Kernel_Function_Execution_Logic_Comparison"  width="500px" >


本章将以Add矢量算子为例，详细介绍如何构建一个完整的CPU侧验证工程，包括工程目录结构、各文件的作用以及核心代码的实现。首先，我们通过目录结构来了解整个工程的组成，明确每个文件的职责，为后续深入代码讲解打下基础。

### 2.1 算子描述

- 算子功能：

    Add算子实现了两个数据相加，返回相加结果的功能。对应的数学表达式为：

    `z = x + y`

    本样例中实现的是固定shape为8*2048的Add算子，同时添加printf命令用于打印指定的内容。

- 算子规格：  
  Add算子：  
  <table style="margin: 0; margin-right: auto;">
  <tr><td rowspan="1" align="center">算子类型(OpType)</td><td colspan="4" align="center">Add</td></tr>
  </tr>
  <tr><td rowspan="3" align="center">算子输入</td><td align="center">name</td><td align="center">shape</td><td align="center">data type</td><td align="center">format</td></tr>
  <tr><td align="center">x</td><td align="center">8 * 2048</td><td align="center">half</td><td align="center">ND</td></tr>
  <tr><td align="center">y</td><td align="center">8 * 2048</td><td align="center">half</td><td align="center">ND</td></tr>
  </tr>
  </tr>
  <tr><td rowspan="1" align="center">算子输出</td><td align="center">z</td><td align="center">8 * 2048</td><td align="center">half</td><td align="center">ND</td></tr>
  </tr>
  <tr><td rowspan="1" align="center">核函数名</td><td colspan="4" align="center">add_custom</td></tr>
  </table>

### 2.1 代码目录结构说明

CPU侧验证程序工程目录结构如下：
```
cpu_debug/
├── scripts/
│   ├── gen_data.py              # 输入数据和真值数据生成脚本
│   └── verify_result.py         # 验证输出数据与真值一致性的脚本
├── CMakeLists.txt               # CMake编译配置文件
├── data_utils.h                 # 数据读写工具函数
└── add.cpp                      # 主程序，包含矢量算子核函数实现和CPU域的算子调用
```

在接下来的小节中，我们将逐一剖析每个文件的具体内容与编写要点，帮助开发者掌握在CPU仿真环境下验证AscendC算子的完整流程。

执行如下命令，创建工程目录：

In [3]:
!mkdir -p Sources/07.02/cpu_debug/scripts

### 2.2 add.cpp文件详解

add.cpp 是整个CPU侧验证工程的核心文件，它包含了头文件引入与常量定义、算子核函数实现以及CPU侧主程序调用三大部分。下面将逐一剖析。

#### 2.2.1 头文件引入与常量定义

- **头文件说明**：
    - kernel_operator.h：AscendC算子开发的核心头文件，提供了向量计算、数据搬运、流水线管理等基础API。
    - data_utils.h：自定义的数据工具头文件，包含 ReadFile、WriteFile 等辅助函数，用于文件读写。
    - tikicpulib.h：CPU调试库头文件，仅在CPU仿真环境下使用。

- **常量定义**：这些常量决定了数据规模和计算切分方式，是理解核函数逻辑的关键。
    - TOTAL_LENGTH 为输入输出数据的总元素个数（half类型，每个占2字节）。
    - USE_CORE_NUM 指定模拟使用的AI Core数量（这里为8核），实际CPU调试中会创建8个线程模拟并行。
    - BLOCK_LENGTH 为每个AICore需要处理的数据元素个数。
    - TILE_NUM 和 BUFFER_NUM 共同决定了Double Buffer机制下的流水线分段数量。

In [ ]:
%%writefile Sources/07.02/cpu_debug/add.cpp

#include "kernel_operator.h"
#include "data_utils.h"
#include "tikicpulib.h"

constexpr int32_t TOTAL_LENGTH = 8 * 2048;                            // total length of data
constexpr int32_t USE_CORE_NUM = 8;                                   // num of core used
constexpr int32_t BLOCK_LENGTH = TOTAL_LENGTH / USE_CORE_NUM;         // length computed of each core
constexpr int32_t TILE_NUM = 8;                                       // split data into 1 tiles for each core
constexpr int32_t BUFFER_NUM = 2;                                     // tensor num for each queue
constexpr int32_t TILE_LENGTH = BLOCK_LENGTH / TILE_NUM / BUFFER_NUM; // separate to 2 parts, due to double buffer

#### 2.2.2 算子核函数实现

算子核函数部分由 `KernelAdd` 类和核函数入口 `add_custom` 组成，完整实现了 Add 算子的计算逻辑。由于矢量 Add 算子的核函数实现已在第二章中详细介绍，此处不再赘述。

In [ ]:
%%writefile -a Sources/07.02/cpu_debug/add.cpp

class KernelAdd {
public:
    __aicore__ inline KernelAdd() {}
    __aicore__ inline void Init(GM_ADDR x, GM_ADDR y, GM_ADDR z)
    {
        xGm.SetGlobalBuffer((__gm__ half *)x + BLOCK_LENGTH * AscendC::GetBlockIdx(), BLOCK_LENGTH);
        yGm.SetGlobalBuffer((__gm__ half *)y + BLOCK_LENGTH * AscendC::GetBlockIdx(), BLOCK_LENGTH);
        zGm.SetGlobalBuffer((__gm__ half *)z + BLOCK_LENGTH * AscendC::GetBlockIdx(), BLOCK_LENGTH);
        pipe.InitBuffer(inQueueX, BUFFER_NUM, TILE_LENGTH * sizeof(half));
        pipe.InitBuffer(inQueueY, BUFFER_NUM, TILE_LENGTH * sizeof(half));
        pipe.InitBuffer(outQueueZ, BUFFER_NUM, TILE_LENGTH * sizeof(half));
    }
    __aicore__ inline void Process()
    {
        int32_t loopCount = TILE_NUM * BUFFER_NUM;
        for (int32_t i = 0; i < loopCount; i++) {
            CopyIn(i);
            Compute(i);
            CopyOut(i);
        }
    }

private:
    __aicore__ inline void CopyIn(int32_t progress)
    {
        AscendC::LocalTensor<half> xLocal = inQueueX.AllocTensor<half>();
        AscendC::LocalTensor<half> yLocal = inQueueY.AllocTensor<half>();
        AscendC::DataCopy(xLocal, xGm[progress * TILE_LENGTH], TILE_LENGTH);
        AscendC::DataCopy(yLocal, yGm[progress * TILE_LENGTH], TILE_LENGTH);
        inQueueX.EnQue(xLocal);
        inQueueY.EnQue(yLocal);
    }
    __aicore__ inline void Compute(int32_t progress)
    {
        AscendC::LocalTensor<half> xLocal = inQueueX.DeQue<half>();
        AscendC::LocalTensor<half> yLocal = inQueueY.DeQue<half>();
        AscendC::LocalTensor<half> zLocal = outQueueZ.AllocTensor<half>();
        AscendC::Add(zLocal, xLocal, yLocal, TILE_LENGTH);
        outQueueZ.EnQue<half>(zLocal);
        inQueueX.FreeTensor(xLocal);
        inQueueY.FreeTensor(yLocal);
    }
    __aicore__ inline void CopyOut(int32_t progress)
    {
        AscendC::LocalTensor<half> zLocal = outQueueZ.DeQue<half>();
        AscendC::DataCopy(zGm[progress * TILE_LENGTH], zLocal, TILE_LENGTH);
        outQueueZ.FreeTensor(zLocal);
    }

private:
    AscendC::TPipe pipe;
    AscendC::TQue<AscendC::TPosition::VECIN, BUFFER_NUM> inQueueX, inQueueY;
    AscendC::TQue<AscendC::TPosition::VECOUT, BUFFER_NUM> outQueueZ;
    AscendC::GlobalTensor<half> xGm;
    AscendC::GlobalTensor<half> yGm;
    AscendC::GlobalTensor<half> zGm;
};

__global__ __aicore__ void add_custom(GM_ADDR x, GM_ADDR y, GM_ADDR z)
{
    KernelAdd op;
    op.Init(x, y, z);
    op.Process();
}


#### 2.2.3 主程序CPU侧运行验证

main函数是CPU调试的主入口，完成算子核函数CPU侧运行验证的步骤如下：

<img src="./images/CPU-side_Verification_Steps.png" alt="CPU-side_Verification_Steps"  width="200px" >

下面列出了本节代码中使用的CPU域调试相关接口及其功能，方便开发者对照参考：

<table border="1" cellpadding="8" cellspacing="0" style="text-align: left; margin-left: 0;">
  <thead>
    <tr>
      <th>接口/宏</th>
      <th>功能说明</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <td>GmAlloc</td>
      <td>进行核函数的CPU侧运行验证时，用于创建共享内存：在/tmp目录下创建一个共享文件，并返回该文件的映射指针。</td>
    </tr>
    <tr>
      <td>GmFree</td>
      <td>进行核函数的CPU侧运行验证时，用于释放通过GmAlloc申请的共享内存。</td>
    </tr>
    <tr>
      <td>SetKernelMode</td>
      <td>针对分离模式，CPU调测时，设置内核模式为单AIV模式，单AIC模式或者MIX模式，以分别支持单AIV矢量算子，单AIC矩阵算子，MIX混合算子的CPU调试。不调用该接口的情况下，默认为MIX模式。为保证算子代码在多个硬件平台兼容，耦合模式下也可以调用，该场景下接口不会生效，不影响正常调试。</td>
    </tr>
    <tr>
      <td>ICPU_RUN_KF</td>
      <td>进行核函数的CPU侧运行验证时，CPU调测总入口，完成CPU侧的算子程序调用。</td>
    </tr>
  </tbody>
</table>



In [ ]:
%%writefile -a Sources/07.02/cpu_debug/add.cpp

int32_t main(int32_t argc, char *argv[])
{
    uint32_t numBlocks = 8;
    size_t inputByteSize = 8 * 2048 * sizeof(uint16_t);
    size_t outputByteSize = 8 * 2048 * sizeof(uint16_t);

    // 使用GmAlloc分配共享内存，并进行数据初始化
    uint8_t *x = (uint8_t *)AscendC::GmAlloc(inputByteSize);
    uint8_t *y = (uint8_t *)AscendC::GmAlloc(inputByteSize);
    uint8_t *z = (uint8_t *)AscendC::GmAlloc(outputByteSize);

    ReadFile("../input_x.bin", inputByteSize, x, inputByteSize);
    ReadFile("../input_y.bin", inputByteSize, y, inputByteSize);

    // 矢量算子需要设置内核模式为AIV模式
    AscendC::SetKernelMode(KernelMode::AIV_MODE);
    // 调用ICPU_RUN_KF调测宏，完成核函数CPU侧的调用
    ICPU_RUN_KF(add_custom, numBlocks, x, y, z); // use this macro for cpu debug

    // 输出数据写出
    WriteFile("../output.bin", z, outputByteSize);

    // 调用GmFree释放申请的资源
    AscendC::GmFree((void *)x);
    AscendC::GmFree((void *)y);
    AscendC::GmFree((void *)z);
    return 0;
}

### 2.3 data_utils.h工具函数

data_utils.h 是工程中的辅助工具头文件，主要封装了文件读写操作以及日志打印宏，用于简化数据准备与结果验证流程。该文件在 CPU 侧调试和 NPU 侧运行中均可使用，提供了一致的接口。

In [ ]:
%%writefile Sources/07.02/cpu_debug/data_utils.h

#ifndef DATA_UTILS_H
#define DATA_UTILS_H

#include <fcntl.h>
#include <sys/stat.h>
#include <unistd.h>

#include <cassert>
#include <cstdio>
#include <fstream>
#include <iomanip>
#include <iostream>
#include <string>
#include <vector>

#include "acl/acl.h"

// 打印 INFO 级别日志
#define INFO_LOG(fmt, args...) fprintf(stdout, "[INFO]  " fmt "\n", ##args)
// 打印 WARN 级别日志
#define WARN_LOG(fmt, args...) fprintf(stdout, "[WARN]  " fmt "\n", ##args)
// 打印 ERROR 级别日志
#define ERROR_LOG(fmt, args...) fprintf(stdout, "[ERROR]  " fmt "\n", ##args)

// 检查 AscendCL 接口返回值，出错时打印错误信息
#define CHECK_ACL(x)                                                                        \
    do {                                                                                    \
        aclError __ret = x;                                                                 \
        if (__ret != ACL_ERROR_NONE) {                                                      \
            std::cerr << __FILE__ << ":" << __LINE__ << " aclError:" << __ret << std::endl; \
        }                                                                                   \
    } while (0);

// 从二进制文件读取数据到缓冲区
bool ReadFile(const std::string &filePath, size_t &fileSize, void *buffer, size_t bufferSize)
{
    struct stat sBuf;
    int fileStatus = stat(filePath.data(), &sBuf);
    if (fileStatus == -1) {
        ERROR_LOG("failed to get file");
        return false;
    }
    if (S_ISREG(sBuf.st_mode) == 0) {
        ERROR_LOG("%s is not a file, please enter a file", filePath.c_str());
        return false;
    }

    std::ifstream file;
    file.open(filePath, std::ios::binary);
    if (!file.is_open()) {
        ERROR_LOG("Open file failed. path = %s", filePath.c_str());
        return false;
    }

    std::filebuf *buf = file.rdbuf();
    size_t size = buf->pubseekoff(0, std::ios::end, std::ios::in);
    if (size == 0) {
        ERROR_LOG("file size is 0");
        file.close();
        return false;
    }
    if (size > bufferSize) {
        ERROR_LOG("file size is larger than buffer size");
        file.close();
        return false;
    }
    buf->pubseekpos(0, std::ios::in);
    buf->sgetn(static_cast<char *>(buffer), size);
    fileSize = size;
    file.close();
    return true;
}

// 将缓冲区数据写入二进制文件
bool WriteFile(const std::string &filePath, const void *buffer, size_t size)
{
    if (buffer == nullptr) {
        ERROR_LOG("Write file failed. buffer is nullptr");
        return false;
    }

    int fd = open(filePath.c_str(), O_RDWR | O_CREAT | O_TRUNC, S_IRUSR | S_IWRITE);
    if (fd < 0) {
        ERROR_LOG("Open file failed. path = %s", filePath.c_str());
        return false;
    }

    size_t writeSize = write(fd, buffer, size);
    (void)close(fd);
    if (writeSize != size) {
        ERROR_LOG("Write file Failed.");
        return false;
    }

    return true;
}

// 打印任意类型数组，支持指定每行元素个数
template <typename T> void DoPrintData(const T *data, size_t count, size_t elementsPerRow)
{
    assert(elementsPerRow != 0);
    for (size_t i = 0; i < count; ++i) {
        std::cout << std::setw(10) << data[i];
        if (i % elementsPerRow == elementsPerRow - 1) {
            std::cout << std::endl;
        }
    }
}

// 打印 aclFloat16 类型数组（转换为 float 后打印），支持指定每行元素个数
void DoPrintHalfData(const aclFloat16 *data, size_t count, size_t elementsPerRow)
{
    assert(elementsPerRow != 0);
    for (size_t i = 0; i < count; ++i) {
        std::cout << std::setw(10) << std::setprecision(6) << aclFloat16ToFloat(data[i]);
        if (i % elementsPerRow == elementsPerRow - 1) {
            std::cout << std::endl;
        }
    }
}

#endif // DATA_UTILS_H

### 2.4 准备输入数据

在 CPU 侧运行验证之前，需要准备输入数据以及用于比对的真值数据（golden data）。本节将使用 Python 脚本 `gen_data.py` 生成符合算子输入要求的随机数据，并将其保存为二进制文件。具体代码如下：

In [ ]:
%%writefile Sources/07.02/cpu_debug/scripts/gen_data.py

import numpy as np

def gen_golden_data_simple():
    # 生成输入 x：范围 [1, 100)，形状 (8, 2048)，数据类型 float16
    input_x = np.random.uniform(1, 100, [8, 2048]).astype(np.float16)
    # 生成输入 y：范围 [1, 100)，形状 (8, 2048)，数据类型 float16
    input_y = np.random.uniform(1, 100, [8, 2048]).astype(np.float16)
    # 计算加法结果作为真值数据（golden）
    golden = (input_x + input_y).astype(np.float16)

    # 保存输入 x 到二进制文件
    input_x.tofile("./input_x.bin")
    # 保存输入 y 到二进制文件
    input_y.tofile("./input_y.bin")
    # 保存真值到二进制文件
    golden.tofile("./golden.bin")

    print("="*50)
    print("数据生成执行完成！")
    print(f"已生成文件：input_x.bin、input_y.bin、golden.bin")
    print(f"数据形状：(8, 2048)，数据类型：float16")
    print("="*50)
    
if __name__ == "__main__":
    # 脚本直接执行时，调用数据生成函数
    gen_golden_data_simple()


执行以下命令即可生成所需的数据文件：

In [ ]:
!cd Sources/07.02/cpu_debug && python3 ./scripts/gen_data.py

### 2.5 编译运行

按照以下步骤编译并执行算子验证程序：

In [ ]:
%%writefile Sources/07.02/cpu_debug/CMakeLists.txt

cmake_minimum_required(VERSION 3.16)
project(cpudebug)

# 查找并加载 tikicpulib（CPU 调试必备），若找不到则终止配置
find_package(tikicpulib REQUIRED)
add_executable(add ${CMAKE_CURRENT_SOURCE_DIR}/add.cpp)
target_compile_options(add PRIVATE
    -g -O2 -std=c++17 -D_GLIBCXX_USE_CXX11_ABI=0 -Wall -Werror
)
# 链接所需库：
target_link_libraries(add PRIVATE
    ascendcl
    tikicpulib::${soc_version}
)

install(TARGETS add
    LIBRARY DESTINATION ${CMAKE_INSTALL_LIBDIR}
    ARCHIVE DESTINATION ${CMAKE_INSTALL_LIBDIR}
    RUNTIME DESTINATION ${CMAKE_INSTALL_BINDIR}
)

In [ ]:
!cd Sources/07.02/cpu_debug && mkdir -p build && cd build && cmake .. -Dsoc_version=Ascend910B4 && make -j 
!cd Sources/07.02/cpu_debug/build && ./add

### 2.6 验证输出结果

完成算子运行后，需要将实际输出与真值数据进行比对，以验证计算结果的正确性。本节将使用 `scripts/verify_result.py` 脚本完成这一验证工作。该脚本通过逐元素比较两个二进制文件的内容，输出误差统计信息并返回最终的验证结果。

In [ ]:
%%writefile Sources/07.02/cpu_debug/scripts/verify_result.py

import sys
import numpy as np

# 设置比较的容差参数（针对 float16 类型）
relative_tol = 1e-3   # 相对误差容忍度
absolute_tol = 1e-5   # 绝对误差容忍度
error_tol = 1e-3      # 允许的最大误差比例

def verify_result(output, golden):
    """
    验证算子输出与真值数据是否一致。
    参数:
        output: 算子输出文件路径
        golden: 真值数据文件路径
    返回:
        bool: 若误差比例不超过 error_tol 返回 True，否则 False
    """
    # 从二进制文件读取数据并转换为一维数组
    output = np.fromfile(output, dtype=np.float16).reshape(-1)
    golden = np.fromfile(golden, dtype=np.float16).reshape(-1)
    
    # 逐元素比较，考虑容差和 NaN 相等的情况
    different_element_results = np.isclose(output,
                                           golden,
                                           rtol=relative_tol,
                                           atol=absolute_tol,
                                           equal_nan=True)
    # 获取不一致元素的索引
    different_element_indexes = np.where(different_element_results == False)[0]
    # 打印前 100 个不一致元素的信息
    for index in range(len(different_element_indexes)):
        real_index = different_element_indexes[index]
        golden_data = golden[real_index]
        output_data = output[real_index]
        print(
            "data index: %06d, expected: %-.9f, actual: %-.9f, rdiff: %-.6f" %
            (real_index, golden_data, output_data,
             abs(output_data - golden_data) / golden_data))
        if index == 100:
            break
    # 计算不一致元素的比例
    error_ratio = float(different_element_indexes.size) / golden.size
    print("error ratio: %.4f, tolerance: %.4f" % (error_ratio, error_tol))
    # 判断误差比例是否在允许范围内
    return error_ratio <= error_tol


if __name__ == '__main__':
    try:
        res = verify_result(sys.argv[1], sys.argv[2])
        if not res:
            raise ValueError("[ERROR] result error")
        else:
            print("test pass!")
    except Exception as e:
        print(e)
        sys.exit(1)


执行以下命令即可完成结果验证：

In [ ]:
!cd Sources/07.02/cpu_debug && python3 scripts/verify_result.py output.bin golden.bin

执行上述命令后，终端输出如下结果，表明精度对比成功：
```
test pass!
```

---

## 3. 基于printf的CPU域调试实践

在算子开发的 CPU 域调试阶段，printf 接口是核心的格式化输出工具，能够实现日志信息的精准打印，帮助开发者快速定位代码执行中的问题。该接口同时支持 CPU 域与 NPU 域调试场景，在 CPU 域调试时，其使用逻辑与 C/C++ 标准printf高度兼容，且针对算子开发场景做了适配性扩展。

### 3.1 接口原型与基础功能

算子 kernel 侧的printf调试接口提供两个等效的函数原型，均支持变参输入，具体定义如下：

```
template <class... Args>
__aicore__ inline void printf(__gm__ const char* fmt, Args&&... args);

template <class... Args>
__aicore__ inline void PRINTF(__gm__ const char* fmt, Args&&... args);
```

二者功能完全一致，仅名称存在差异，开发者可根据编码习惯自由选择。接口的核心作用是提供CPU域/NPU域调试场景下的格式化输出功能，适用于算子 kernel 代码中任意需要输出日志的位置，是 CPU 域调试中追踪变量值、验证执行逻辑的关键手段。

### 3.2 基础使用示例

在算子kernel侧核函数的实现代码中，开发者可以通过调用 printf 接口在需要的位置输出日志信息，以辅助调试。以下示例展示了如何在 Add 算子的 Init 函数中添加 printf 语句，打印当前 AI Core 的逻辑 ID 以及每个核心需要处理的数据长度 BLOCK_LENGTH：

In [ ]:
%%writefile Sources/07.02/cpu_debug/add.cpp

#include "kernel_operator.h"
#include "data_utils.h"
#include "tikicpulib.h"

constexpr int32_t TOTAL_LENGTH = 8 * 2048;                            // total length of data
constexpr int32_t USE_CORE_NUM = 8;                                   // num of core used
constexpr int32_t BLOCK_LENGTH = TOTAL_LENGTH / USE_CORE_NUM;         // length computed of each core
constexpr int32_t TILE_NUM = 8;                                       // split data into 1 tiles for each core
constexpr int32_t BUFFER_NUM = 2;                                     // tensor num for each queue
constexpr int32_t TILE_LENGTH = BLOCK_LENGTH / TILE_NUM / BUFFER_NUM; // separate to 2 parts, due to double buffer
constexpr int32_t OFFSET_LENGTH = 32; // offset length for DumpAccChkPoint
constexpr int32_t DUMP_LENGTH = 32; // dump length

class KernelAdd {
public:
    __aicore__ inline KernelAdd() {}
    __aicore__ inline void Init(GM_ADDR x, GM_ADDR y, GM_ADDR z)
    {
        xGm.SetGlobalBuffer((__gm__ half *)x + BLOCK_LENGTH * AscendC::GetBlockIdx(), BLOCK_LENGTH);
        yGm.SetGlobalBuffer((__gm__ half *)y + BLOCK_LENGTH * AscendC::GetBlockIdx(), BLOCK_LENGTH);
        zGm.SetGlobalBuffer((__gm__ half *)z + BLOCK_LENGTH * AscendC::GetBlockIdx(), BLOCK_LENGTH);
        pipe.InitBuffer(inQueueX, BUFFER_NUM, TILE_LENGTH * sizeof(half));
        pipe.InitBuffer(inQueueY, BUFFER_NUM, TILE_LENGTH * sizeof(half));
        pipe.InitBuffer(outQueueZ, BUFFER_NUM, TILE_LENGTH * sizeof(half));
        // 添加打印调试信息
        AscendC::printf("[Block (%lu/%lu)]: BLOCK_LENGTH is %d\n", AscendC::GetBlockIdx(), AscendC::GetBlockNum(), BLOCK_LENGTH);
    }
    __aicore__ inline void Process()
    {
        int32_t loopCount = TILE_NUM * BUFFER_NUM;
        for (int32_t i = 0; i < loopCount; i++) {
            CopyIn(i);
            Compute(i);
            CopyOut(i);
        }
    }

private:
    __aicore__ inline void CopyIn(int32_t progress)
    {
        AscendC::LocalTensor<half> xLocal = inQueueX.AllocTensor<half>();
        AscendC::LocalTensor<half> yLocal = inQueueY.AllocTensor<half>();
        AscendC::DataCopy(xLocal, xGm[progress * TILE_LENGTH], TILE_LENGTH);
        AscendC::DataCopy(yLocal, yGm[progress * TILE_LENGTH], TILE_LENGTH);
        inQueueX.EnQue(xLocal);
        inQueueY.EnQue(yLocal);
    }
    __aicore__ inline void Compute(int32_t progress)
    {
        AscendC::LocalTensor<half> xLocal = inQueueX.DeQue<half>();
        AscendC::LocalTensor<half> yLocal = inQueueY.DeQue<half>();
        AscendC::LocalTensor<half> zLocal = outQueueZ.AllocTensor<half>();
        AscendC::Add(zLocal, xLocal, yLocal, TILE_LENGTH);
        outQueueZ.EnQue<half>(zLocal);
        inQueueX.FreeTensor(xLocal);
        inQueueY.FreeTensor(yLocal);
    }
    __aicore__ inline void CopyOut(int32_t progress)
    {
        AscendC::LocalTensor<half> zLocal = outQueueZ.DeQue<half>();
        AscendC::DataCopy(zGm[progress * TILE_LENGTH], zLocal, TILE_LENGTH);
        outQueueZ.FreeTensor(zLocal);
    }

private:
    AscendC::TPipe pipe;
    AscendC::TQue<AscendC::TPosition::VECIN, BUFFER_NUM> inQueueX, inQueueY;
    AscendC::TQue<AscendC::TPosition::VECOUT, BUFFER_NUM> outQueueZ;
    AscendC::GlobalTensor<half> xGm;
    AscendC::GlobalTensor<half> yGm;
    AscendC::GlobalTensor<half> zGm;
};

__global__ __aicore__ void add_custom(GM_ADDR x, GM_ADDR y, GM_ADDR z)
{
    KernelAdd op;
    op.Init(x, y, z);
    op.Process();
}

int32_t main(int32_t argc, char *argv[])
{
    uint32_t numBlocks = 8;
    size_t inputByteSize = 8 * 2048 * sizeof(uint16_t);
    size_t outputByteSize = 8 * 2048 * sizeof(uint16_t);

    uint8_t *x = (uint8_t *)AscendC::GmAlloc(inputByteSize);
    uint8_t *y = (uint8_t *)AscendC::GmAlloc(inputByteSize);
    uint8_t *z = (uint8_t *)AscendC::GmAlloc(outputByteSize);

    ReadFile("../input_x.bin", inputByteSize, x, inputByteSize);
    ReadFile("../input_y.bin", inputByteSize, y, inputByteSize);

    AscendC::SetKernelMode(KernelMode::AIV_MODE);
    ICPU_RUN_KF(add_custom, numBlocks, x, y, z); // use this macro for cpu debug

    WriteFile("../output.bin", z, outputByteSize);

    AscendC::GmFree((void *)x);
    AscendC::GmFree((void *)y);
    AscendC::GmFree((void *)z);
    return 0;
}


执行以下命令，重新编译并运行算子，即可查看调试信息：

In [ ]:
!cd Sources/07.02/cpu_debug && mkdir -p build && cd build && cmake .. -Dsoc_version=Ascend910B4 && make -j 
!cd Sources/07.02/cpu_debug/build && ./add

---
## 课后实践

**前提条件**：已按照本章内容完成所有工程文件的编写。

**实践要求**：修改 cpu_debug 工程中的 add.cpp 文件，在核函数的 Compute 函数中添加 printf 打印语句，在每个AICore第一次调用Compute函数时打印当前核逻辑ID、xLocal 和 yLocal 两个Tensor的长度以及TILE_LENGTH。编译运行程序，观察控制台输出的日志信息，验证程序执行流程是否符合预期。

### 1. 添加打印信息

在如下代码框中修改添加打印信息：

In [ ]:
%%writefile Sources/07.02/cpu_debug/add.cpp

#include "kernel_operator.h"
#include "data_utils.h"
#include "tikicpulib.h"

constexpr int32_t TOTAL_LENGTH = 8 * 2048;                            // total length of data
constexpr int32_t USE_CORE_NUM = 8;                                   // num of core used
constexpr int32_t BLOCK_LENGTH = TOTAL_LENGTH / USE_CORE_NUM;         // length computed of each core
constexpr int32_t TILE_NUM = 8;                                       // split data into 1 tiles for each core
constexpr int32_t BUFFER_NUM = 2;                                     // tensor num for each queue
constexpr int32_t TILE_LENGTH = BLOCK_LENGTH / TILE_NUM / BUFFER_NUM; // separate to 2 parts, due to double buffer
constexpr int32_t OFFSET_LENGTH = 32; // offset length for DumpAccChkPoint
constexpr int32_t DUMP_LENGTH = 32; // dump length

class KernelAdd {
public:
    __aicore__ inline KernelAdd() {}
    __aicore__ inline void Init(GM_ADDR x, GM_ADDR y, GM_ADDR z)
    {
        xGm.SetGlobalBuffer((__gm__ half *)x + BLOCK_LENGTH * AscendC::GetBlockIdx(), BLOCK_LENGTH);
        yGm.SetGlobalBuffer((__gm__ half *)y + BLOCK_LENGTH * AscendC::GetBlockIdx(), BLOCK_LENGTH);
        zGm.SetGlobalBuffer((__gm__ half *)z + BLOCK_LENGTH * AscendC::GetBlockIdx(), BLOCK_LENGTH);
        pipe.InitBuffer(inQueueX, BUFFER_NUM, TILE_LENGTH * sizeof(half));
        pipe.InitBuffer(inQueueY, BUFFER_NUM, TILE_LENGTH * sizeof(half));
        pipe.InitBuffer(outQueueZ, BUFFER_NUM, TILE_LENGTH * sizeof(half));
    }
    __aicore__ inline void Process()
    {
        int32_t loopCount = TILE_NUM * BUFFER_NUM;
        for (int32_t i = 0; i < loopCount; i++) {
            CopyIn(i);
            Compute(i);
            CopyOut(i);
        }
    }

private:
    __aicore__ inline void CopyIn(int32_t progress)
    {
        AscendC::LocalTensor<half> xLocal = inQueueX.AllocTensor<half>();
        AscendC::LocalTensor<half> yLocal = inQueueY.AllocTensor<half>();
        AscendC::DataCopy(xLocal, xGm[progress * TILE_LENGTH], TILE_LENGTH);
        AscendC::DataCopy(yLocal, yGm[progress * TILE_LENGTH], TILE_LENGTH);
        inQueueX.EnQue(xLocal);
        inQueueY.EnQue(yLocal);
    }
    __aicore__ inline void Compute(int32_t progress)
    {
        AscendC::LocalTensor<half> xLocal = inQueueX.DeQue<half>();
        AscendC::LocalTensor<half> yLocal = inQueueY.DeQue<half>();
        AscendC::LocalTensor<half> zLocal = outQueueZ.AllocTensor<half>();
        // 此处添加打印调试信息……

        AscendC::Add(zLocal, xLocal, yLocal, TILE_LENGTH);
        outQueueZ.EnQue<half>(zLocal);
        inQueueX.FreeTensor(xLocal);
        inQueueY.FreeTensor(yLocal);
    }
    __aicore__ inline void CopyOut(int32_t progress)
    {
        AscendC::LocalTensor<half> zLocal = outQueueZ.DeQue<half>();
        AscendC::DataCopy(zGm[progress * TILE_LENGTH], zLocal, TILE_LENGTH);
        outQueueZ.FreeTensor(zLocal);
    }

private:
    AscendC::TPipe pipe;
    AscendC::TQue<AscendC::TPosition::VECIN, BUFFER_NUM> inQueueX, inQueueY;
    AscendC::TQue<AscendC::TPosition::VECOUT, BUFFER_NUM> outQueueZ;
    AscendC::GlobalTensor<half> xGm;
    AscendC::GlobalTensor<half> yGm;
    AscendC::GlobalTensor<half> zGm;
};

__global__ __aicore__ void add_custom(GM_ADDR x, GM_ADDR y, GM_ADDR z)
{
    KernelAdd op;
    op.Init(x, y, z);
    op.Process();
}

int32_t main(int32_t argc, char *argv[])
{
    uint32_t numBlocks = 8;
    size_t inputByteSize = 8 * 2048 * sizeof(uint16_t);
    size_t outputByteSize = 8 * 2048 * sizeof(uint16_t);

    uint8_t *x = (uint8_t *)AscendC::GmAlloc(inputByteSize);
    uint8_t *y = (uint8_t *)AscendC::GmAlloc(inputByteSize);
    uint8_t *z = (uint8_t *)AscendC::GmAlloc(outputByteSize);

    ReadFile("../input_x.bin", inputByteSize, x, inputByteSize);
    ReadFile("../input_y.bin", inputByteSize, y, inputByteSize);

    AscendC::SetKernelMode(KernelMode::AIV_MODE);
    ICPU_RUN_KF(add_custom, numBlocks, x, y, z); // use this macro for cpu debug

    WriteFile("../output.bin", z, outputByteSize);

    AscendC::GmFree((void *)x);
    AscendC::GmFree((void *)y);
    AscendC::GmFree((void *)z);
    return 0;
}


### 2. 编译运行

执行以下命令编译并运行算子，查看调试信息（无需修改任何代码）：

In [ ]:
!cd Sources/07.02/cpu_debug && mkdir -p build && cd build && cmake .. -Dsoc_version=Ascend910B4 && make -j 
!cd Sources/07.02/cpu_debug/build && ./add

### 3. 验证结果

预期执行效果如下：

<img src="./images/Result_Pic_0702.png" alt="Result_Pic_0702"  width="500px" >

执行以下代码获取答案。

In [ ]:
!cat ./answer/07.02_answer.txt